# Import Libraries
The following libraries will be used for data cleaning and visualizations.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from ipywidgets import interact, Dropdown


## 1. Collecting and Cleaning Data

**Dataset source**:
1. Download the **Games.csv** file from Kaggle (free with an account). This file has real attendance, real scores, and arenas — perfect for analysis.
2. Link: https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores?select=Games.csv
3. After download is complete, save it in the same folder as your notebook.
4. I also included a copy of the dataset in this repo called `Games.csv`.


In [2]:
df = pd.read_csv('Games.csv', low_memory=False)

# Rename columns for clarity
df = df.rename(columns={
    'hometeamName': 'Home_Team',
    'awayteamName': 'Away_Team',
    'homeScore': 'Home_PTS',
    'awayScore': 'Away_PTS',
    'attendance': 'Attendance'
})

# Keep only regular-season games and drop any rows missing key data
df = df[df['gameType'].str.contains('Regular Season', na=False)]
df = df.dropna(subset=['Attendance', 'Home_PTS', 'Away_PTS'])
df['Home_Team'] = df['Home_Team'].str.strip()
df['Away_Team'] = df['Away_Team'].str.strip()

# Create key columns
df['Home_Win'] = df['Home_PTS'] > df['Away_PTS']
df['Point_Diff'] = df['Home_PTS'] - df['Away_PTS']
df['Home_Win_Pct'] = df.groupby('Home_Team')['Home_Win'].transform('mean')

# Arena coordinates for travel distance
arena_coords = {
    'Suns': (33.4457, -112.0713), 'Celtics': (42.3662, -71.0621), 'Knicks': (40.7505, -73.9934),
    'Trail Blazers': (45.5316, -122.6668), 'Lakers': (34.0430, -118.2673), 'Bucks': (43.0436, -87.9172),
    'Raptors': (43.6435, -79.3791), 'Wizards': (38.8981, -77.0209), 'Hornets': (35.2251, -80.8392),
    'Nets': (40.6827, -73.9753), 'Hawks': (33.7580, -84.3963), 'Heat': (25.7814, -80.1881),
    'Magic': (28.5392, -81.3832), '76ers': (39.9012, -75.1719), 'Pacers': (39.7640, -86.1555),
    'Pistons': (42.3410, -83.0550), 'Bulls': (41.8911, -87.6742), 'Cavaliers': (41.4966, -81.6882),
    'Warriors': (37.7680, -122.3877), 'Clippers': (34.0430, -118.2673), 'Kings': (38.5802, -121.4998),
    'Spurs': (29.4268, -98.4375), 'Jazz': (40.7681, -111.9010), 'Thunder': (35.4634, -97.5150),
    'Pelicans': (29.9490, -90.0527), 'Grizzlies': (35.1380, -90.0500), 'Rockets': (29.7508, -95.3622),
    'Mavericks': (32.7906, -96.8103), 'Nuggets': (39.7483, -105.0077), 'Timberwolves': (44.9743, -93.2580)
}

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['Travel_Distance'] = 0.0
for idx, row in df.iterrows():
    away_home = arena_coords.get(row['Away_Team'])
    game_arena = arena_coords.get(row['Home_Team'])
    if away_home and game_arena:
        df.at[idx, 'Travel_Distance'] = haversine(away_home[0], away_home[1], game_arena[0], game_arena[1])

# Buckets
df['Attendance_Level'] = pd.qcut(df['Attendance'], q=3, labels=['Low', 'Medium', 'High'])
df['Travel_Bucket'] = pd.cut(df['Travel_Distance'], bins=[0, 1000, 2000, 5000], labels=['Short', 'Medium', 'Long'])

## 2. The Interactive Dashboard (All charts update together)

Run the cell below. Choose any team — three visualizations update instantly while the bar chart always shows every team for easy comparison.

In [3]:
# Interactive Dashboard
def update_dashboard(team='All Teams'):
    # Filter for scatter / box / violin
    temp = df.copy()
    if team != 'All Teams':
        temp = temp[(temp['Home_Team'] == team) | (temp['Away_Team'] == team)]
    
    # 1. Grouped Bar - Does not change always shows ALL teams
    win_pct = df.groupby('Home_Team').agg(Home_Win_Pct=('Home_Win', 'mean')).reset_index()
    win_pct['Away_Win_Pct'] = 1 - win_pct['Home_Win_Pct']
    fig1 = px.bar(win_pct, x='Home_Team', y=['Home_Win_Pct', 'Away_Win_Pct'], barmode='group',
                  title='Home vs Away Win % (All Teams)', labels={'value': 'Win %'})
    
    # 2. Scatter - Attendance vs Point Differential
    fig2 = px.scatter(temp, x='Attendance', y='Point_Diff', color='Home_Win',
                     hover_data=['Home_Team', 'Away_Team'],
                     title='Attendance vs Point Differential (color = Home Win)',
                     color_discrete_map={True: 'green', False: 'red'})
    
    # 3. Box Plot - Point Differential by Travel Bucket
    away_games = temp[temp['Travel_Bucket'].notna()]
    fig3 = px.box(away_games, x='Travel_Bucket', y='Point_Diff', color='Travel_Bucket',
                  title='Away Performance by Travel Distance')
    
    # 4. Violin Plot - Win Margin by Crowd Size
    fig4 = px.violin(temp, x='Attendance_Level', y='Point_Diff', color='Home_Win',
                     title='Win Margin by Crowd Size (Home vs Away)')
    
    fig1.show()
    fig2.show()
    fig3.show()
    fig4.show()

team_dropdown = Dropdown(options=['All Teams'] + sorted(df['Home_Team'].dropna().unique()), value='All Teams', description='Team:')
interact(update_dashboard, team=team_dropdown);

interactive(children=(Dropdown(description='Team:', options=('All Teams', '76ers', 'Bucks', 'Bulls', 'Cavalier…